# KonfAI Segmentation Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vboussot/KonfAI/blob/main/examples/Segmentation/Segmentation_demo.ipynb)

This notebook is meant to work from a **fresh environment**, including **Google Colab**.

It will help you:

- bootstrap KonfAI if the runtime is empty
- download the public `Segmentation` demo subset
- inspect one segmentation case
- prepare the standard `TRAIN -> PREDICTION -> EVALUATION` workflow
- optionally launch the commands step by step

The expected layout is one folder per case with `CT.mha` and `SEG.mha`.
The demo `SEG.mha` files are multiclass label maps with labels `0..40`.
        


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys


def find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "examples").exists():
            return candidate
    return None


IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/vboussot/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = find_repo_root(Path.cwd().resolve())
    if REPO_DIR is None:
        raise RuntimeError(
            "Could not locate the KonfAI repository. Run this notebook from the repo or open it in Google Colab."
        )

EXAMPLE_DIR = REPO_DIR / "examples" / "Segmentation"
DATASET_DIR = EXAMPLE_DIR / "Dataset"
print("Repository:", REPO_DIR)
print("Example:", EXAMPLE_DIR)
print("Dataset:", DATASET_DIR)

required_packages = {
    "konfai": str(REPO_DIR),
    "SimpleITK": "SimpleITK",
    "huggingface_hub": "huggingface_hub",
    "matplotlib": "matplotlib",
}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]

if missing_packages:
    print("Installing missing dependencies:", ", ".join(missing_packages))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
else:
    print("KonfAI environment already available.")


## 1. Download the public segmentation demo subset

The cell below downloads only the `Segmentation` subset from the public Hugging Face demo dataset with the `hf` CLI, then removes the local Hugging Face metadata cache created under `Dataset/.cache`.
        


In [ ]:
import shutil

RUN_DOWNLOAD = True

if RUN_DOWNLOAD:
    subprocess.run([
        "hf",
        "download",
        "VBoussot/konfai-demo",
        "--repo-type",
        "dataset",
        "--include",
        "Segmentation/**",
        "--local-dir",
        str(DATASET_DIR),
    ], check=True)

nested_dir = DATASET_DIR / "Segmentation"
if nested_dir.exists():
    for item in nested_dir.iterdir():
        target = DATASET_DIR / item.name
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        shutil.move(str(item), str(target))
    shutil.rmtree(nested_dir)

shutil.rmtree(DATASET_DIR / ".cache", ignore_errors=True)

cases = sorted([path for path in DATASET_DIR.iterdir() if path.is_dir()]) if DATASET_DIR.exists() else []
print("Dataset ready:", DATASET_DIR.exists())
print("Number of cases:", len(cases))
if cases:
    print("First case:", cases[0].name)
        


## 2. Inspect one case

This sanity check prints simple statistics for `CT` and `SEG`, reports which labels are present in the first segmentation map, and displays a representative slice.


In [ ]:
import matplotlib.pyplot as plt
import SimpleITK as sitk
import numpy as np


def describe_image(path: Path):
    image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(image)
    return image, array, {
        "shape": tuple(array.shape),
        "dtype": str(array.dtype),
        "min": float(np.min(array)),
        "max": float(np.max(array)),
        "spacing": tuple(float(x) for x in image.GetSpacing()),
    }

cases = sorted([path for path in DATASET_DIR.iterdir() if path.is_dir()]) if DATASET_DIR.exists() else []
if not cases:
    raise RuntimeError("Dataset is empty. Run the download cell above first.")

case = cases[0]
print("Sample case:", case.name)
_, ct_array, ct_stats = describe_image(case / "CT.mha")
_, seg_array, seg_stats = describe_image(case / "SEG.mha")
print("CT", ct_stats)
print("SEG", seg_stats)
print("Labels present in sample:", np.unique(seg_array)[:50])

if seg_array.ndim >= 3:
    scores = (seg_array > 0).reshape(seg_array.shape[0], -1).sum(axis=1)
    slice_index = int(np.argmax(scores)) if np.any(scores) else int(seg_array.shape[0] // 2)
else:
    slice_index = 0

ct_slice = ct_array[slice_index] if ct_array.ndim >= 3 else ct_array
seg_slice = seg_array[slice_index] if seg_array.ndim >= 3 else seg_array

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(ct_slice, cmap="gray")
axes[0].set_title(f"CT - slice {slice_index}")
axes[0].axis("off")

seg_im = axes[1].imshow(seg_slice, cmap="nipy_spectral", interpolation="nearest", vmin=0, vmax=40)
axes[1].set_title(f"SEG - slice {slice_index}")
axes[1].axis("off")

axes[2].imshow(ct_slice, cmap="gray")
axes[2].imshow(seg_slice, cmap="nipy_spectral", interpolation="nearest", alpha=(seg_slice > 0) * 0.6, vmin=0, vmax=40)
axes[2].set_title("CT + SEG overlay")
axes[2].axis("off")

fig.colorbar(seg_im, ax=axes[1:], fraction=0.025, pad=0.02)
plt.tight_layout()
plt.show()


## 3. Review the workflow files and commands

This baseline uses:

- `Config.yml` for training
- `Prediction.yml` for inference
- `Evaluation.yml` for Dice evaluation

Training uses `CrossEntropyLoss`, while final quality is measured with Dice on labels `1..40`.
        


In [ ]:
import torch


def latest_checkpoint(train_name: str):
    checkpoint_dir = EXAMPLE_DIR / "Checkpoints" / train_name
    checkpoints = sorted(checkpoint_dir.glob("*.pt"), key=lambda path: path.stat().st_mtime) if checkpoint_dir.exists() else []
    return checkpoints[-1] if checkpoints else None

for name in ["Config.yml", "Prediction.yml", "Evaluation.yml"]:
    print(name, "->", EXAMPLE_DIR / name)

device_args = ["--gpu", "0"] if torch.cuda.is_available() else ["--cpu", "1"]
checkpoint = latest_checkpoint("SEG_BASELINE")
checkpoint_arg = str(checkpoint.relative_to(EXAMPLE_DIR)) if checkpoint else "Checkpoints/SEG_BASELINE/<checkpoint>.pt"

TRAIN_CMD = ["konfai", "TRAIN", "-y", *device_args, "--config", "Config.yml"]
PRED_CMD = ["konfai", "PREDICTION", "-y", *device_args, "--config", "Prediction.yml", "--models", checkpoint_arg]
EVAL_CMD = ["konfai", "EVALUATION", "-y", "--config", "Evaluation.yml"]

for name, cmd in [("TRAIN", TRAIN_CMD), ("PREDICTION", PRED_CMD), ("EVALUATION", EVAL_CMD)]:
    print(name, " ".join(cmd))
        


## 4. Optionally run the workflow

Keep all flags set to `False` if you only want to inspect the setup.

If you enable prediction, the notebook will automatically reuse the newest checkpoint from `Checkpoints/SEG_BASELINE/`.
        


In [ ]:
RUN_TRAIN = False
RUN_PREDICTION = False
RUN_EVALUATION = False


def run_command(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=EXAMPLE_DIR)

if RUN_TRAIN:
    run_command(TRAIN_CMD)

if RUN_PREDICTION:
    checkpoint = latest_checkpoint("SEG_BASELINE")
    if checkpoint is None:
        raise RuntimeError("No checkpoint found in Checkpoints/SEG_BASELINE. Run training first.")
    run_command(["konfai", "PREDICTION", "-y", *device_args, "--config", "Prediction.yml", "--models", str(checkpoint.relative_to(EXAMPLE_DIR))])

if RUN_EVALUATION:
    run_command(EVAL_CMD)
        


## 5. Expected outputs

After a complete run, you should find:

- `Checkpoints/SEG_BASELINE/`
- `Statistics/SEG_BASELINE/`
- `Predictions/SEG_BASELINE/`
- `Evaluations/SEG_BASELINE/`

Once the baseline runs, the usual next steps are to tune patch size, channels, augmentations, and label-specific Dice reporting for your own dataset.
        
